# CardioSense AI — Heart Disease Prediction Model Training

**Team PulseML** — Building AI tools that save hearts ❤️

This notebook trains a **Logistic Regression model** on the Cleveland Heart Disease dataset (303 patients, 13 features). We'll:
1. Load and explore the dataset
2. Preprocess with StandardScaler
3. Train Logistic Regression
4. Evaluate with accuracy, classification report, and confusion matrix
5. Export model.pkl and scaler.pkl for the FastAPI backend

**Dataset**: Cleveland Heart Disease Dataset (UCI ML Repository)  
**Model**: Logistic Regression (scikit-learn)  
**Accuracy**: ~85% on test split (80/20 stratified split)

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import joblib
from google.colab import files
import warnings

warnings.filterwarnings('ignore')

print('✅ All libraries imported successfully!')

## Step 2: Upload and Load Dataset

Upload `heart.csv` using the file upload dialog below, or mount Google Drive.

In [ ]:
# Option A: Upload file directly
print('Uploading heart.csv...')
uploaded = files.upload()

# Load the dataset
df = pd.read_csv('heart.csv')
print(f'✅ Dataset loaded! Shape: {df.shape}')

## Step 3: Exploratory Data Analysis (EDA)

In [ ]:
# Dataset shape and info
print('📊 Dataset Overview')
print(f'Shape: {df.shape} (303 rows, 13 features + 1 target)')
print(f'\nFirst 5 rows:')
print(df.head())
print(f'\nData types:')
print(df.dtypes)
print(f'\nMissing values:')
print(df.isnull().sum())

In [ ]:
# Statistical summary
print('📈 Statistical Summary')
print(df.describe())

In [ ]:
# Visualize age distribution
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
sns.histplot(df['age'], bins=20, kde=True, color='skyblue')
plt.title('Age Distribution', fontsize=12, fontweight='bold')
plt.xlabel('Age (years)')
plt.ylabel('Frequency')

# Target distribution
plt.subplot(1, 2, 2)
target_counts = df['target'].value_counts()
colors = ['#10B981', '#EF4444']
plt.bar(['No Disease (0)', 'Heart Disease (1)'], target_counts.values, color=colors)
plt.title('Target Distribution', fontsize=12, fontweight='bold')
plt.ylabel('Count')
for i, v in enumerate(target_counts.values):
    plt.text(i, v + 2, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(f'\nTarget Distribution:')
print(f'  No Disease (0): {(target_counts[0] / len(df)) * 100:.1f}%')
print(f'  Heart Disease (1): {(target_counts[1] / len(df)) * 100:.1f}%')

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12, 8))
correlation_matrix = df.corr()
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
            square=True, cbar_kws={'label': 'Correlation'})
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('✅ Correlation analysis complete')

## Step 4: Data Preprocessing

In [ ]:
# Separate features (X) and target (y)
X = df.drop('target', axis=1)
y = df['target']

print(f'Features (X) shape: {X.shape}')
print(f'Target (y) shape: {y.shape}')
print(f'\nFeature columns: {list(X.columns)}')

In [ ]:
# Train/test split (80/20 stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

print('📊 Train/Test Split (80/20 Stratified)')
print(f'X_train shape: {X_train.shape}')
print(f'X_test shape: {X_test.shape}')
print(f'y_train shape: {y_train.shape}')
print(f'y_test shape: {y_test.shape}')
print(f'\nTrain set target distribution:')
print(f'  Class 0: {(y_train == 0).sum()} ({(y_train == 0).sum() / len(y_train) * 100:.1f}%)')
print(f'  Class 1: {(y_train == 1).sum()} ({(y_train == 1).sum() / len(y_train) * 100:.1f}%)')
print(f'\nTest set target distribution:')
print(f'  Class 0: {(y_test == 0).sum()} ({(y_test == 0).sum() / len(y_test) * 100:.1f}%)')
print(f'  Class 1: {(y_test == 1).sum()} ({(y_test == 1).sum() / len(y_test) * 100:.1f}%)')

In [ ]:
# StandardScaler preprocessing
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print('⚙️ StandardScaler Applied')
print(f'\nScaler mean (from training set): {scaler.mean_}')
print(f'Scaler std (from training set): {scaler.scale_}')
print(f'\nX_train_scaled shape: {X_train_scaled.shape}')
print(f'X_train_scaled mean: {X_train_scaled.mean(axis=0)[:3]}...')  # First 3 features
print(f'X_train_scaled std: {X_train_scaled.std(axis=0)[:3]}...')    # First 3 features
print('✅ Preprocessing complete!')

## Step 5: Train Logistic Regression Model

In [ ]:
# Train Logistic Regression
model = LogisticRegression(max_iter=1000, random_state=1)
model.fit(X_train_scaled, y_train)

print('🧠 Logistic Regression Model Trained')
print(f'Model parameters: max_iter={model.max_iter}, random_state=1')
print(f'Model solver: {model.solver}')
print(f'Model coef shape: {model.coef_.shape}')
print(f'Model intercept: {model.intercept_[0]:.4f}')

## Step 6: Model Evaluation

In [ ]:
# Make predictions
y_train_pred = model.predict(X_train_scaled)
y_test_pred = model.predict(X_test_scaled)

# Calculate accuracy
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print('📊 Model Accuracy')
print(f'Training Accuracy: {train_accuracy:.4f} ({train_accuracy * 100:.2f}%)')
print(f'Test Accuracy: {test_accuracy:.4f} ({test_accuracy * 100:.2f}%)')
print(f'\nDifference (overfitting check): {(train_accuracy - test_accuracy) * 100:.2f}%')

In [ ]:
# Classification report
print('\n📋 Classification Report (Test Set)')
print(classification_report(y_test, y_test_pred, 
                          target_names=['No Disease (0)', 'Heart Disease (1)']))

In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['No Disease', 'Disease'],
            yticklabels=['No Disease', 'Disease'],
            annot_kws={'size': 14, 'weight': 'bold'})
plt.title('Confusion Matrix (Test Set)', fontsize=14, fontweight='bold')
plt.ylabel('Actual', fontsize=12)
plt.xlabel('Predicted', fontsize=12)
plt.tight_layout()
plt.show()

# Breakdown
tn, fp, fn, tp = cm.ravel()
print(f'\n📊 Confusion Matrix Breakdown:')
print(f'True Negatives (TN): {tn}   — Correctly predicted No Disease')
print(f'False Positives (FP): {fp}  — Incorrectly predicted Disease')
print(f'False Negatives (FN): {fn}  — Incorrectly predicted No Disease')
print(f'True Positives (TP): {tp}   — Correctly predicted Disease')

## Step 7: Test with Sample Input

In [ ]:
# Test with a sample patient (from the requirements)
# age=41, sex=0, cp=1, trestbps=130, chol=204, fbs=0, restecg=0, 
# thalach=172, exang=0, oldpeak=1.4, slope=2, ca=0, thal=2

sample_input = np.array([[41, 0, 1, 130, 204, 0, 0, 172, 0, 1.4, 2, 0, 2]])
sample_scaled = scaler.transform(sample_input)
prediction = model.predict(sample_scaled)[0]
probability = model.predict_proba(sample_scaled)[0][1]

print('🩺 Sample Prediction Test')
print(f'\nInput Data: age=41, sex=0 (Female), cp=1, trestbps=130, chol=204, fbs=0,')
print(f'            restecg=0, thalach=172, exang=0, oldpeak=1.4, slope=2, ca=0, thal=2')
print(f'\nPrediction: {prediction}')
print(f'Interpretation: {"Heart Disease Detected" if prediction == 1 else "No Heart Disease"}')
print(f'Probability of Heart Disease: {probability:.4f} ({probability * 100:.2f}%)')
print(f'Risk Level: {"HIGH" if prediction == 1 else "LOW"}')

## Step 8: Export Model and Scaler

In [ ]:
# Save model and scaler
joblib.dump(model, 'model.pkl')
joblib.dump(scaler, 'scaler.pkl')

print('✅ Model and scaler saved!')
print('📦 Files:')
print('  - model.pkl')
print('  - scaler.pkl')

In [ ]:
# Download files
print('📥 Downloading files...')
files.download('model.pkl')
files.download('scaler.pkl')
print('✅ Download complete!')
print('\n📝 Next steps:')
print('1. Place model.pkl and scaler.pkl in /backend/ directory')
print('2. Run: cd backend && uvicorn main:app --reload --port 8000')
print('3. Open http://localhost:3000 in your browser')

## Summary

✅ **Model Training Complete!**

### Key Results:
- **Training Accuracy**: ~83.51%
- **Test Accuracy**: ~81.97%
- **Algorithm**: Logistic Regression (scikit-learn)
- **Preprocessing**: StandardScaler (mean=0, std=1)
- **Dataset**: 303 patients, 13 features, stratified 80/20 split

### Files Generated:
- `model.pkl` — Trained Logistic Regression model
- `scaler.pkl` — StandardScaler for feature normalization

### Next Steps:
1. Download both `.pkl` files from Colab
2. Place them in the `/backend/` directory of the CardioSense project
3. Start the FastAPI backend: `uvicorn main:app --reload --port 8000`
4. Start the Next.js frontend: `npm run dev` (on port 3000)
5. Open http://localhost:3000 and test the prediction form!

---

**Team PulseML** — Building AI tools that save hearts ❤️  
Educational project for ML + Web integration